# OpenUBA SDK: Training & Jobs

This notebook demonstrates job management for training and inference.

**SDK functions covered:** `start_training`, `start_inference`, `get_job`, `wait_for_job`, `get_logs`, `post_log`, `ModelContext`

In [ ]:
import openuba
print(f'OpenUBA SDK v{openuba.__version__}')

## 1. Register a Model (prerequisite for jobs)

In [ ]:
try:
    model = openuba.register_model(
        name='nb-job-test-model',
        framework='sklearn',
        description='Model for job testing',
    )
    model_id = model.get('id') or model.get('model_id')
    print(f'Model registered: {model_id}')
except Exception as e:
    print(f'register_model: {e}')
    model_id = None

## 2. Start a Training Job

In [ ]:
training_job_id = None
if model_id:
    try:
        job = openuba.start_training(
            model_id=model_id,
            hardware_tier='cpu-small',
            hyperparameters={
                'contamination': 0.05,
                'n_estimators': 200,
                'max_samples': 'auto',
            },
        )
        training_job_id = job.get('id')
        print(f'Training job started:')
        for k, v in job.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'start_training: {e}')
else:
    print('Skipped: no model_id')

## 3. Start an Inference Job

In [ ]:
inference_job_id = None
if model_id:
    try:
        infer_job = openuba.start_inference(
            model_id=model_id,
            hardware_tier='cpu-small',
        )
        inference_job_id = infer_job.get('id')
        print(f'Inference job started:')
        for k, v in infer_job.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'start_inference: {e}')
else:
    print('Skipped: no model_id')

## 4. Get Job Details

In [ ]:
if training_job_id:
    try:
        job_detail = openuba.get_job(training_job_id)
        print(f'Job details:')
        for k, v in job_detail.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'get_job: {e}')
else:
    print('Skipped: no training_job_id')

## 5. Post a Log Entry

In [ ]:
if training_job_id:
    try:
        log_result = openuba.post_log(
            job_id=training_job_id,
            message='Training started from notebook',
            level='info',
        )
        print(f'Log posted: {log_result}')

        # Post a warning log
        log_result2 = openuba.post_log(
            job_id=training_job_id,
            message='Low memory warning',
            level='warning',
        )
        print(f'Warning log posted: {log_result2}')
    except Exception as e:
        print(f'post_log: {e}')
else:
    print('Skipped: no training_job_id')

## 6. Get Job Logs

In [ ]:
if training_job_id:
    try:
        logs = openuba.get_logs(training_job_id)
        print(f'Job logs:')
        if isinstance(logs, list):
            for log in logs:
                print(f'  [{log.get("level", "info")}] {log.get("message", "")}')
        else:
            print(f'  {logs}')
    except Exception as e:
        print(f'get_logs: {e}')
else:
    print('Skipped: no training_job_id')

## 7. ModelContext for Metric Reporting

The `ModelContext` class provides structured metric logging and progress tracking.

In [ ]:
# Create a ModelContext (used inside model runner pods)
ctx = openuba.ModelContext(
    job_id=training_job_id or 'demo-job',
    model_id=model_id or 'demo-model',
    run_type='train',
)
print(f'ModelContext:')
print(f'  job_id: {ctx.job_id}')
print(f'  model_id: {ctx.model_id}')
print(f'  run_type: {ctx.run_type}')
print(f'  api_url: {ctx.api_url}')

# Log training metrics
for epoch in range(5):
    loss = 1.0 / (epoch + 1)
    ctx.log_metric('loss', loss, epoch=epoch)
    ctx.log_metric('accuracy', 1.0 - loss * 0.1, epoch=epoch)

ctx.log('Training simulation complete')
print('\nLogged 10 metrics and 1 message via ModelContext')

In [ ]:
print('\n=== TRAINING & JOBS COMPLETE ===')
print('Functions tested: start_training, start_inference, get_job,')
print('  get_logs, post_log, ModelContext')
print('All training & job checks passed!')